# KNN on the 63-column NSCG + O*NET integration

`nscg2023_onet30_2_integration63.csv` has the same **74,016 respondents in 123 occupation categories** as the PR #17 subset (first 50 columns and row order identical), plus 13 original NSCG fields: `SALARY`, `HRSWK`, `EMSECSM` (sector), `EMSIZE` (employer size) and nine job-facet satisfaction ratings (`SATADV` ... `SATSOC`).

The evaluation follows `Forest.ipynb`:

* target: satisfied = `JOBSATIS` 1 or 2,
* 5 folds grouped by occupation (here `ONET_PROFILE_ID`, which is one-to-one with `N3OCPR`), saved to `folds_expanded.csv`,
* the same metrics and occupation-level bootstrap interval.

The first five feature sets repeat the PR #17 run exactly, so the numbers can be compared. The new sets add the job characteristics, and, separately, the facet satisfaction ratings.

**About the facet ratings:** they ask how satisfied the person is with salary, challenge, location and so on, at the same time as the overall `JOBSATIS` question. They are parts of the outcome, not causes of it, so a model using them is not predicting dissatisfaction from person-job fit. They are reported only as an upper bound, not as a candidate model.

In [1]:
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_score,
                             recall_score, f1_score)
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
DATA_PATH = "nscg2023_onet30_2_integration63.csv"
FOLDS_PATH = "folds_expanded.csv"
K_GRID = [25, 50, 100, 200, 400, 800, 1600, 3200]

**Part 1: Setup**

In [2]:
df = pd.read_csv(DATA_PATH, low_memory=False)
y = df["JOBSATIS"].isin([1, 2]).astype(int).values
groups = df["ONET_PROFILE_ID"].values
assert df.groupby("ONET_PROFILE_ID")["N3OCPR"].nunique().max() == 1
print("shape:", df.shape)
print(f"satisfied: {y.mean():.1%} | unsatisfied: {1 - y.mean():.1%} | occupations: {len(np.unique(groups))}")
print(df["MAPPING_TYPE"].value_counts().to_dict())

shape: (74016, 63)
satisfied: 90.1% | unsatisfied: 9.9% | occupations: 123
{'group_mean_partial': 33658, 'single_SOC': 18859, 'group_mean_complete': 17170, 'single_SOC_broader_profile': 3010, 'single_available_proxy': 1274, 'residual_single_proxy': 45}


**Part 2: Features** (fit gaps use the same definitions as the earlier KNN notebooks)

gap = importance to the person (0 to 1) − how much the occupation offers it (0 to 1). The NSCG-to-O\*NET matching is our assumption, not an official crosswalk.

| O\*NET Work Value | Matched NSCG items |
|---|---|
| Achievement | FACCHAL |
| Independence | FACIND |
| Recognition | FACADV, FACRESP |
| Working conditions | FACSAL, FACBEN, FACSEC, FACLOC |
| Relationships | FACSOC |

In [3]:
FAC = ["FACADV", "FACBEN", "FACCHAL", "FACIND", "FACLOC", "FACRESP", "FACSAL", "FACSEC", "FACSOC"]
WV = ["ONET_WV_achievement", "ONET_WV_working_conditions", "ONET_WV_recognition",
      "ONET_WV_relationships", "ONET_WV_support", "ONET_WV_independence"]
FIT_MAP = {"achievement": ["FACCHAL"], "independence": ["FACIND"],
           "recognition": ["FACADV", "FACRESP"],
           "working_conditions": ["FACSAL", "FACBEN", "FACSEC", "FACLOC"],
           "relationships": ["FACSOC"]}
importance = (4 - df[FAC]) / 3
for wv_name, items in FIT_MAP.items():
    df[f"GAP_{wv_name}"] = importance[items].mean(axis=1) - (df[f"ONET_WV_{wv_name}"] - 1) / 6
GAP = [f"GAP_{k}" for k in FIT_MAP]
assert not df[FAC + WV + GAP].isna().any().any()

JOB_NUM = ["LOG_SALARY", "HRSWK"]
JOB = JOB_NUM + ["EMSECSM", "EMSIZE"]
SAT = ["SATADV", "SATBEN", "SATCHAL", "SATIND", "SATLOC", "SATRESP", "SATSAL", "SATSEC", "SATSOC"]
df["LOG_SALARY"] = np.log1p(df["SALARY"])   # salary is very skewed, so it is log-scaled
df["EMSECSM"] = df["EMSECSM"].astype(str)    # category codes, not amounts
df["EMSIZE"] = df["EMSIZE"].astype(str)
assert not df[["SALARY", "HRSWK"] + SAT].isna().any().any()

NUM_BASE = ["AGE"]
BACKGROUND = NUM_BASE + ["SEX_2023", "DGRDG", "OCEDRLP", "SUPWK", "WAPRI"]
FEATURE_SETS = {
    "demographics_only":          NUM_BASE + ["SEX_2023", "DGRDG"],
    "background_full":            BACKGROUND,
    "background_+_values":        BACKGROUND + FAC,
    "background_+_values_+_onet": BACKGROUND + FAC + WV,
    "background_+_fit_gaps":      BACKGROUND + GAP,
    # new: job characteristics from the 13 added fields
    "background_+_job":                BACKGROUND + JOB,
    "background_+_values_+_job":       BACKGROUND + FAC + JOB,
    "background_+_job_+_fit_gaps":     BACKGROUND + JOB + GAP,
    # upper bound only (facet ratings are part of the outcome)
    "UPPER_BOUND_background_+_facet_sat": BACKGROUND + SAT,
}
NUMERIC = set(NUM_BASE + FAC + WV + GAP + JOB_NUM + SAT)
df["DGRDG"] = df["DGRDG"].astype(str)
df["WAPRI"] = df["WAPRI"].astype(str)

**Part 3: Occupation-grouped folds**

Created once with the same recipe as `Forest.ipynb` (`StratifiedGroupKFold`, 5 splits, seed 42) and saved, so other models can reuse exactly the same split.

In [4]:
if Path(FOLDS_PATH).exists():
    saved = pd.read_csv(FOLDS_PATH)
    fold_id = df[["OBSNUM"]].merge(saved, on="OBSNUM", how="left", validate="one_to_one")["fold"].values.astype(int)
    print("loaded", FOLDS_PATH)
else:
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    fold_id = np.empty(len(df), dtype=int)
    for k, (_, te) in enumerate(cv.split(df, y, groups)):
        fold_id[te] = k
    pd.DataFrame({"OBSNUM": df["OBSNUM"], "fold": fold_id}).to_csv(FOLDS_PATH, index=False)
    print("created", FOLDS_PATH)
folds = [(np.where(fold_id != k)[0], np.where(fold_id == k)[0]) for k in range(5)]
print("fold sizes:      ", [len(te) for _, te in folds])
print("unsatisfied/fold:", [int((1 - y[te]).sum()) for _, te in folds])
print("occupations/fold:", [len(np.unique(groups[te])) for _, te in folds])

loaded folds_expanded.csv
fold sizes:       [14806, 14624, 14990, 14852, 14744]
unsatisfied/fold: [1471, 1484, 1464, 1470, 1472]
occupations/fold: [25, 24, 24, 24, 26]


**Part 4: KNN, with k chosen inside the training folds only**

For speed, neighbors are found once (up to the largest k), and the score for every k is the share of satisfied people among the first k neighbors. This gives the same predictions as scikit-learn's `KNeighborsClassifier` with uniform weights, without refitting for each k. `k` is picked by an inner 4-fold grouped CV on the training occupations, so the test fold is never used to choose it.

In [5]:
def encoder(cols):
    num = [c for c in cols if c in NUMERIC]
    cat = [c for c in cols if c not in NUMERIC]
    return ColumnTransformer([("num", StandardScaler(), num),
                              ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])

def scores_for_all_k(cols, tr, te, ks=K_GRID):
    enc = encoder(cols).fit(df.iloc[tr][cols])
    Xtr, Xte = enc.transform(df.iloc[tr][cols]), enc.transform(df.iloc[te][cols])
    kmax = min(max(ks), len(tr))
    _, idx = NearestNeighbors(n_neighbors=kmax, n_jobs=-1).fit(Xtr).kneighbors(Xte)
    cum = np.cumsum(y[tr][idx], axis=1)
    return {k: cum[:, k - 1] / k for k in ks if k <= kmax}

def choose_k(cols, tr):
    inner = GroupKFold(n_splits=4)
    aucs = {k: [] for k in K_GRID}
    for itr, ite in inner.split(tr, y[tr], groups[tr]):
        s = scores_for_all_k(cols, tr[itr], tr[ite])
        for k, p in s.items():
            aucs[k].append(roc_auc_score(y[tr[ite]], p))
    return max(aucs, key=lambda k: np.mean(aucs[k]))

**Part 5: Metrics** (same as the forest)

Precision, recall and F1 flag someone as unsatisfied when their score is below the training fold's share of satisfied people (KNN has no class weights). ROC AUC and PR AUC use no cut-off.

In [6]:
def metrics(y, p, thr):
    pu, yu, pred_u = 1 - p, 1 - y, (p < thr).astype(int)
    return {"roc_auc": roc_auc_score(y, p),
            "pr_auc_unsat": average_precision_score(yu, pu),
            "precision_unsat": precision_score(yu, pred_u, zero_division=0),
            "recall_unsat": recall_score(yu, pred_u),
            "f1_unsat": f1_score(yu, pred_u)}

def cluster_boot_ci(y, p, groups, n=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    ug = np.unique(groups)
    idx = {g: np.where(groups == g)[0] for g in ug}
    out = []
    for _ in range(n):
        ii = np.concatenate([idx[g] for g in rng.choice(ug, len(ug))])
        if len(np.unique(y[ii])) == 2:
            out.append(roc_auc_score(y[ii], p[ii]))
    return np.percentile(out, [2.5, 97.5])

**Part 6: Fit and compare**

In [7]:
rows, preds, chosen_k = [], {"OBSNUM": df["OBSNUM"], "y_satisfied": y, "fold": fold_id}, {}

def summarize(name, p, thr, with_ci=True):
    r = {"model": name, **metrics(y, p, thr)}
    r["auc_ci_low"], r["auc_ci_high"] = cluster_boot_ci(y, p, groups) if with_ci else (np.nan, np.nan)
    fa = [roc_auc_score(y[te], p[te]) for _, te in folds]
    r["fold_auc_mean"], r["fold_auc_sd"] = np.mean(fa), np.std(fa)
    rows.append(r)

thr = np.zeros(len(df)); p_dummy = np.zeros(len(df))
for tr, te in folds:
    thr[te] = y[tr].mean(); p_dummy[te] = y[tr].mean()
summarize("dummy_prior", p_dummy, thr, with_ci=False)

for name, cols in FEATURE_SETS.items():
    p, ks = np.zeros(len(df)), []
    for tr, te in folds:
        k = choose_k(cols, tr); ks.append(k)
        p[te] = scores_for_all_k(cols, tr, te, ks=[k])[k]
    preds[f"knn_{name}"] = p; chosen_k[name] = ks
    summarize(f"knn_{name}", p, thr)

results = pd.DataFrame(rows).round(3)
results.to_csv("knn_integration63_results.csv", index=False)
pd.DataFrame(preds).to_csv("knn_integration63_oof_predictions.csv", index=False)
print("k chosen in each fold:", chosen_k)
results

k chosen in each fold: {'demographics_only': [1600, 3200, 3200, 1600, 3200], 'background_full': [800, 3200, 1600, 800, 3200], 'background_+_values': [800, 1600, 1600, 800, 1600], 'background_+_values_+_onet': [800, 800, 800, 400, 1600], 'background_+_fit_gaps': [1600, 800, 800, 800, 1600], 'background_+_job': [800, 800, 1600, 800, 800], 'background_+_values_+_job': [800, 800, 800, 800, 800], 'background_+_job_+_fit_gaps': [800, 400, 1600, 400, 800], 'UPPER_BOUND_background_+_facet_sat': [800, 800, 800, 800, 800]}


,model,roc_auc,pr_auc_unsat,precision_unsat,recall_unsat,f1_unsat,auc_ci_low,auc_ci_high,fold_auc_mean,fold_auc_sd
0,dummy_prior,0.496,0.099,0.000,0.000,0.000,NaN,NaN,0.500,0.000
1,knn_demographics_only,0.577,0.124,0.120,0.592,0.200,0.566,0.590,0.578,0.009
2,knn_background_full,0.626,0.156,0.144,0.523,0.226,0.611,0.641,0.626,0.011
3,knn_background_+_values,0.658,0.171,0.162,0.530,0.248,0.648,0.669,0.659,0.007
4,knn_background_+_values_+_onet,0.647,0.168,0.157,0.518,0.240,0.633,0.660,0.648,0.007
5,knn_background_+_fit_gaps,0.640,0.165,0.155,0.488,0.235,0.626,0.653,0.640,0.006
6,knn_background_+_job,0.649,0.176,0.154,0.541,0.240,0.636,0.663,0.650,0.007
7,knn_background_+_values_+_job,0.677,0.187,0.179,0.512,0.265,0.667,0.687,0.677,0.009
8,knn_background_+_job_+_fit_gaps,0.655,0.177,0.165,0.491,0.247,0.644,0.668,0.657,0.007
9,knn_UPPER_BOUND_background_+_facet_sat,0.929,0.635,0.466,0.794,0.588,0.925,0.934,0.930,0.004


**Part 7: How much does k matter?** (a check, not for picking the reported model)

In [8]:
sens = []
for name in ["background_full", "background_+_job"]:
    cols = FEATURE_SETS[name]
    oof = {k: np.zeros(len(df)) for k in K_GRID}
    for tr, te in folds:
        for k, s in scores_for_all_k(cols, tr, te).items():
            oof[k][te] = s
    for k in K_GRID:
        sens.append({"features": name, "k": k, "roc_auc": round(roc_auc_score(y, oof[k]), 3),
                     "pr_auc_unsat": round(average_precision_score(1 - y, 1 - oof[k]), 3)})
sens = pd.DataFrame(sens)
sens.to_csv("knn_integration63_k_sensitivity.csv", index=False)
sens.pivot(index="k", columns="features", values="roc_auc")

features,background_+_job,background_full
k,,
25,0.598,0.579
50,0.614,0.594
100,0.627,0.606
200,0.639,0.614
400,0.646,0.622
800,0.649,0.626
1600,0.649,0.627
3200,0.649,0.627


**Part 8: Does it hold for exact matches only?**

23,249 people are linked through a proxy profile. This re-scores the same out-of-fold predictions on the people with **no proxy** (`MAPPING_IS_PROXY == False`), to check the conclusion isn't driven by the looser mappings.

In [9]:
exact = ~df["MAPPING_IS_PROXY"].astype(str).str.lower().eq("true").values
print(f"exact-match people: {exact.sum():,} of {len(df):,}")
sub = []
for name in FEATURE_SETS:
    p = preds[f"knn_{name}"]
    sub.append({"model": f"knn_{name}",
                "roc_auc_all": round(roc_auc_score(y, p), 3),
                "roc_auc_exact_only": round(roc_auc_score(y[exact], p[exact]), 3),
                "pr_auc_exact_only": round(average_precision_score(1 - y[exact], 1 - p[exact]), 3)})
pd.DataFrame(sub)

exact-match people: 50,767 of 74,016


,model,roc_auc_all,roc_auc_exact_only,pr_auc_exact_only
0,knn_demographics_only,0.577,0.581,0.124
1,knn_background_full,0.626,0.627,0.156
2,knn_background_+_values,0.658,0.654,0.165
3,knn_background_+_values_+_onet,0.647,0.644,0.166
4,knn_background_+_fit_gaps,0.640,0.639,0.165
5,knn_background_+_job,0.649,0.652,0.177
6,knn_background_+_values_+_job,0.677,0.677,0.184
7,knn_background_+_job_+_fit_gaps,0.655,0.656,0.179
8,knn_UPPER_BOUND_background_+_facet_sat,0.929,0.933,0.642
